In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

class Paths:
    p = "/Users/shirokoshikentaro/Desktop/Python3/house-prices-advanced-regression-techniques"
    train = p + "/train.csv"
    test = p + "/test.csv"
    sample = p + "/sample_submission.csv"

# ===== ステップ1: データ読み込み =====
train = pd.read_csv(Paths.train)
test = pd.read_csv(Paths.test)

print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")

# ===== ステップ2: 目的変数を先に取り出す =====
y_train = np.log1p(train["SalePrice"].values)
print(f"y_train shape: {y_train.shape}")

# ===== ステップ3: 新規特徴量作成 =====
for df in [train, test]:
    # 既存の特徴量
    df["QualGrLiv"] = df["OverallQual"] * df["GrLivArea"]
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["QualTotalSF"] = df["OverallQual"] * df["TotalSF"]
    df["Age"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["TotalBath"] = (df["FullBath"] + 0.5*df["HalfBath"].fillna(0) + 
                       df["BsmtFullBath"].fillna(0) + 0.5*df["BsmtHalfBath"].fillna(0))
    df["AreaPerRoom"] = df["GrLivArea"] / df["TotRmsAbvGrd"].replace(0, 1)
    df["GarageScore"] = df["GarageCars"] * df["GarageArea"]

    # log変換した特徴量
    df["Log_GrLivArea"] = np.log1p(df["GrLivArea"])
    df["Log_LotArea"] = np.log1p(df["LotArea"])
    df["Log_TotalSF"] = np.log1p(df["TotalSF"])
    
    # ⭐新規追加：厳選した特徴量のみ
    # 地上階の総面積（品質で重み付け）
    df["QualGrLivLog"] = df["OverallQual"] * df["Log_GrLivArea"]
    
    # 総ポーチ面積
    df["TotalPorchSF"] = (df["OpenPorchSF"] + df["EnclosedPorch"] + 
                          df["3SsnPorch"].fillna(0) + df["ScreenPorch"].fillna(0))
    
    # 建物の新しさスコア
    df["IsNew"] = (df["Age"] <= 5).astype(int)
    df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)
    
    # 地下室の有無と面積の組み合わせ
    df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["BsmtScore"] = df["TotalBsmtSF"] * df["HasBsmt"]

# ===== ステップ4: 特徴量定義 =====
num_features = [
    "LotArea", "YearBuilt", "YearRemodAdd",
    "GrLivArea", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF",
    "FullBath", "BedroomAbvGr", "TotRmsAbvGrd",
    "GarageCars", "GarageArea", "OverallQual", "OverallCond",
    "QualGrLiv", "TotalSF", "QualTotalSF", 
    "Age", "RemodAge", "TotalBath", "AreaPerRoom", "GarageScore",
    "WoodDeckSF", "OpenPorchSF", "EnclosedPorch",
    "Fireplaces", "HalfBath", "BsmtFullBath",
    # ⭐新規特徴量を追加
    "QualGrLivLog", "TotalPorchSF", "IsNew", "IsRemodeled", 
    "HasBsmt", "BsmtScore",
]

cat_features = [
    "Neighborhood", "BldgType", "HouseStyle",
    "MSZoning", "Foundation", "GarageType",
    "ExterQual", "KitchenQual", "BsmtQual",
    "HeatingQC", "FireplaceQu", "GarageFinish",
]

all_features = num_features + cat_features
print(f"総特徴量: {len(all_features)}個")

# ===== ステップ5: 欠損値処理 =====
print("欠損値処理中...")
for col in num_features:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

for col in cat_features:
    train[col] = train[col].fillna("None")
    test[col] = test[col].fillna("None")

# ===== ステップ6: エンコーディング =====
for col in cat_features:
    le = LabelEncoder()
    combined = pd.concat([train[col].astype(str), test[col].astype(str)])
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

# ===== ステップ7: X_train, X_test作成 =====
X_train = train[all_features].values
X_test = test[all_features].values

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")

# ===== ステップ8: CV（⭐0.13574を出したベストパラメータを使用）=====
# 前回最も良かったパラメータを使用してください
params = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": 31,  # 前回のベストを使用
    "learning_rate": 0.05,
    "n_estimators": 10000,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "random_state": 123,
    "verbose": -1,
}

cv = KFold(n_splits=5, shuffle=True, random_state=123)
metrics = []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]
    
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    metrics.append(rmse)
    print(f"Fold {fold}: {rmse:.5f}")

print(f"\n[CV] {np.mean(metrics):.5f}±{np.std(metrics):.5f}")

# ===== ステップ9: 全データで学習 & 予測 =====
model = lgb.LGBMRegressor(**params)

model.fit(X_train, y_train, 
          eval_set=[(X_train, y_train)],
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)])

y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)

# ===== ステップ10: 提出ファイル =====
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": y_pred
})

submission.to_csv("submission_with_features.csv", index=False)
print(f"\n✅ 保存完了！平均予測価格: ${y_pred.mean():,.0f}")

train shape: (1460, 81)
test shape: (1459, 80)
y_train shape: (1460,)
総特徴量: 46個
欠損値処理中...
X_train shape: (1460, 46)
X_test shape: (1459, 46)
y_train shape: (1460,)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[131]	valid_0's rmse: 0.116314
Fold 0: 0.11631
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1478]	valid_0's rmse: 0.138649
Fold 1: 0.13865
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[218]	valid_0's rmse: 0.136313
Fold 2: 0.13631
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[207]	valid_0's rmse: 0.137528
Fold 3: 0.13753
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[140]	valid_0's rmse: 0.1382
Fold 4: 0.13820

[CV] 0.13340±0.00858
[100]	training's rmse: 0.0757279
[200]	training's rmse: 0.052347
[300]	training's rmse: 0.03901